# Proyecto 2: Sistema Multi-Agente ReAct

Flujo completo en Colab (GPU T4) con **arquitectura hexagonal**: el notebook solo conecta adaptadores reales con el núcleo de `src/`.

- **Agentes (LangGraph):** Investigador (bucle ReAct *Thought → Action → Observation*), Auditor de Hechos (guardrail que rechaza y devuelve el flujo) y Redactor.
- **Herramientas:** `kb` (ChromaDB sobre WikiQA), `web` (DuckDuckGo), `calc` (GSM8K) y `python` (MBPP, en un proceso aislado con tiempo límite).
- **Concurrencia:** `kb` y `web` se consultan en paralelo (fan-out de LangGraph) antes del bucle ReAct.
- **Orquestador:** Mistral-7B-Instruct-v0.3 cuantizado a 4 bits corriendo en la T4.
- **Evaluación:** 50 consultas (20 WikiQA + 15 GSM8K + 15 MBPP) contra un Naive RAG de un solo paso, con *Faithfulness* y *Answer Relevance* (LLM-as-a-Judge) y verificación objetiva en GSM8K y MBPP.

El código se toma de `/content/Multi-Agent-ReAct` si ya fue subido (por ejemplo con `colab upload`); si no, se clona de GitHub.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dependencias

In [ ]:
!pip install -q langgraph chromadb ddgs datasets bitsandbytes accelerate

## 3. Código del proyecto

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/RenzoFelixAponte/Multi-Agent-ReAct.git"
for candidate in ("/content/Multi-Agent-ReAct", os.path.abspath(".."), os.getcwd()):
    if os.path.isdir(os.path.join(candidate, "src", "application")):
        PROJ = candidate
        break
else:
    PROJ = "/content/Multi-Agent-ReAct"
    subprocess.run(["git", "clone", "-q", REPO_URL, PROJ], check=True)
sys.path.insert(0, f"{PROJ}/src")
print("Código del proyecto en:", PROJ)

## 4. Configuración y adaptadores reales

- `BACKEND`: orquestador de los agentes. **`"mistral"`** = modelo local de 4 bits en la T4 (el que pide el enunciado); `"nvidia"` = API remota, solo para desarrollo sin GPU.
- `JUDGE_BACKEND`: modelo que actúa de juez. `"same"` usa el mismo modelo que orquesta; `"nvidia"` usa un modelo remoto más grande. **Elección documentada:** por defecto el juez es el mismo Mistral de 7B para que la corrida sea autocontenida en la T4. Un 7B como juez es ruidoso y puede favorecer sus propias respuestas; por eso cada fallo del formato del juez se cuenta (`juez_fallido`) y se puede repetir la evaluación con `JUDGE_BACKEND = "nvidia"`.
- Secretos: `HF_TOKEN` (Mistral es un modelo restringido: hay que aceptar sus términos en Hugging Face) y, si se usa NVIDIA, `NVIDIA_API_KEY`. En Colab van en el panel 🔑 *Secrets*; con la CLI se pasan con `colab exec --env`.

In [ ]:
import os
from adapters.duckduckgo_search import DuckDuckGoSearch
from adapters.chroma_kb import ChromaKB
from adapters.subprocess_code_runner import SubprocessCodeRunner

BACKEND = "mistral"        # "mistral" (local, 4 bits) | "nvidia" (remoto)
JUDGE_BACKEND = "same"     # "same" (el mismo modelo) | "nvidia" (juez remoto)
FULL_RUN = os.environ.get("FULL_RUN", "1") == "1"   # 0 = solo la prueba de tiempo
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

try:  # secretos del panel de Colab; fuera de Colab se usa el .env o las variables de entorno
    from google.colab import userdata
    for name in ("HF_TOKEN", "NVIDIA_API_KEY"):
        try:
            os.environ.setdefault(name, userdata.get(name))
        except Exception:
            pass
except ImportError:
    pass

if BACKEND == "nvidia":
    from adapters.nvidia_llm import NvidiaLLM
    llm = NvidiaLLM()
else:
    from adapters.mistral_llm import MistralLocalLLM
    llm = MistralLocalLLM(MODEL_ID)

if JUDGE_BACKEND == "nvidia":
    from adapters.nvidia_llm import NvidiaLLM
    judge_llm = NvidiaLLM()
else:
    judge_llm = llm

print("Orquestador:", getattr(llm, "model", "local"))
print("Juez:", getattr(judge_llm, "model", "local"))

web = DuckDuckGoSearch()
kb = ChromaKB(path="/content/chroma_db" if os.path.isdir("/content") else "./chroma_db")
runner = SubprocessCodeRunner()

## 5. Indexar WikiQA en ChromaDB

Se indexan **todas** las frases candidatas de cada pregunta, también las incorrectas (`label=0`). Así la base tiene distractores y el Naive RAG no parte con la respuesta servida en bandeja.

In [ ]:
from adapters.hf_dataset_adapter import HFDatasetAdapter
from adapters.hf_mappers import wikiqa_documents

datasets = HFDatasetAdapter()
wiki = datasets.load("wiki_qa", "validation", limit=300)   # 126 preguntas con respuesta correcta
documents = wikiqa_documents(wiki)
kb.index(documents)
print(f"{len(wiki)} consultas, {len(documents)} frases indexadas (correctas y distractoras)")

## 6. Grafo de agentes

Un grafo por tipo de tarea, con las herramientas que le corresponden:

- **WikiQA:** `kb || web` en paralelo → Investigador → Auditor → Redactor.
- **GSM8K:** Investigador con `calc`.
- **MBPP:** Investigador con `python` (ejecución aislada).

Si el Auditor rechaza, el flujo vuelve al Investigador con el motivo (hasta `max_rounds`).

In [ ]:
from IPython.display import Image, display
from application.graph import build_flow

flows = {
    "wiki_qa": build_flow(llm, web, kb, max_rounds=3, prefetch=True),
    "gsm8k": build_flow(llm, web, kb, max_rounds=3),
    "mbpp": build_flow(llm, web, kb, max_rounds=3, runner=runner),
}
try:
    display(Image(flows["wiki_qa"].get_graph().draw_mermaid_png()))
except Exception as e:  # draw_mermaid_png necesita acceso a mermaid.ink
    print("Diagrama PNG no disponible:", e)
    print(flows["wiki_qa"].get_graph().draw_mermaid())

## 7. Una consulta con el flujo completo

In [ ]:
item = wiki[0]
state = flows["wiki_qa"].invoke({"question": item.question})
print("Pregunta:", item.question)
print("Rondas:", state["rounds"], "| aprobado:", state["verdict"].approved)
print("Evidencia:", [(e.source, e.text[:60]) for e in state["report"].evidence])
print("Respuesta:", state.get("final", "[sin respuesta aprobada]"))

## 8. Prueba de tiempo

Antes de lanzar las 50 consultas se miden unas pocas (1 de cada dataset, con ambos métodos) y se extrapola el tiempo total.

In [ ]:
import time
from adapters.jsonl_result_store import JsonlResultStore
from application.eval_set import EVAL_MIX, build_eval_set
from application.eval_summary import latest
from application.evaluator import Evaluator
from application.judge import Judge

RESULTS_DIR = "/content/results" if os.path.isdir("/content") else os.path.join(PROJ, "results")


def make_evaluator(filename):
    store = JsonlResultStore(os.path.join(RESULTS_DIR, filename))
    return Evaluator(flows, llm, kb, Judge(judge_llm), store, runner), store


def show(r):
    print(f"[{r.dataset:7}] {r.item_id:>6} {r.method:12} respondida={r.answered!s:5} "
          f"faith={r.faithfulness} rel={r.relevance} correcta={r.correct} {r.seconds:6.1f}s {r.error}")


smoke_set = build_eval_set(datasets, tuple((n, s, 1) for n, s, _ in EVAL_MIX))
smoke_eval, smoke_store = make_evaluator("prueba_tiempo.jsonl")
smoke_eval.run(smoke_set, on_record=show)

smoke = latest(smoke_store.load())
per_record = sum(r.seconds for r in smoke) / len(smoke)
total = sum(c for _, _, c in EVAL_MIX) * 2 * per_record
print(f"\n~{per_record:.0f} s por registro -> estimado para {sum(c for _, _, c in EVAL_MIX)} consultas x 2 métodos: {total / 60:.0f} min")

## 9. Evaluación sobre 50 consultas

Cada resultado se guarda en `results/evaluacion.jsonl` apenas se calcula. Si la celda se interrumpe, **volver a ejecutarla reanuda** donde quedó (salta lo hecho y reintenta lo que falló).

In [ ]:
eval_set = build_eval_set(datasets)
print(len(eval_set), "consultas:", {d: sum(i.dataset == d for i in eval_set) for d in ("wiki_qa", "gsm8k", "mbpp")})
evaluator, store = make_evaluator("evaluacion.jsonl")
if FULL_RUN:
    evaluator.run(eval_set, on_record=show)
else:
    print("FULL_RUN=0: se omite la corrida completa")

## 10. Tabla comparativa: multi-agente vs Naive RAG

- `faithfulness` y `answer_relevance`: promedio de 0 a 1 que da el juez. Una **abstención** (el Auditor nunca aprobó) puntúa 0 en ambas; `respondidas` indica qué fracción respondió.
- `exactitud`: verificación objetiva, solo en GSM8K (número final) y MBPP (pruebas unitarias). WikiQA no tiene verificación objetiva.
- `errores` y `juez_fallido` cuentan consultas que fallaron y respuestas del juez con formato inválido.

In [ ]:
import pandas as pd
from dataclasses import asdict
from application.eval_summary import summarize

records = store.load()
table = pd.DataFrame(summarize(records))
display(table)

os.makedirs(RESULTS_DIR, exist_ok=True)
table.to_csv(os.path.join(RESULTS_DIR, "tabla_comparativa.csv"), index=False)
pd.DataFrame([asdict(r) for r in latest(records)]).to_csv(
    os.path.join(RESULTS_DIR, "resultados_por_consulta.csv"), index=False)

overall = table[table["dataset"] == "todos"].set_index("método")
if {"multiagente", "naive_rag"} <= set(overall.index):
    for metric in ("faithfulness", "answer_relevance"):
        multi, naive = overall.loc["multiagente", metric], overall.loc["naive_rag", metric]
        verdict = "supera a" if multi > naive else "NO supera a"
        print(f"{metric}: multi-agente {multi} vs Naive RAG {naive} -> el multi-agente {verdict} Naive RAG")

## 11. Notas metodológicas

- **Naive RAG:** recupera 3 fragmentos de la misma base y hace una sola llamada al LLM, sin herramientas ni auditoría. En GSM8K y MBPP la base de WikiQA no aporta contexto útil, así que ahí compara contra un modelo que razona sin herramientas.
- **Juez:** la elección del modelo juez afecta los números (ver sección 4). Las métricas siguen las definiciones de RAGAS pero están implementadas con un único LLM-as-a-Judge, sin la librería `ragas`.
- **MBPP:** la pregunta incluye los `assert` de `test_list`, como en Austin et al. (2021), y la corrección se mide ejecutándolos.
- **Aislamiento del código:** el proceso hijo tiene tiempo, CPU y memoria limitados, pero no bloquea red ni archivos; es seguro solo porque la VM de Colab es desechable.